# PY ASSESMENT SOLUTION NOTEBOOK

In [1]:
# Numerical arrays
import numpy as np

#Math
import math

#C-like structs
import struct

# sys: checking the machine's native byte order (Problem 1F)
import sys

#For refernecingn
import hashlib


import types

## Problem 1: Representing SHA-256 Data

SHA-256 is a cryptographic hash function that takes an input message of any length and produces a fixed size 256 bits (32 bytes) output.

### Sub-section A: 32-bit words

32-bit word is a number represented using 32 binary digits. It can hold a value from 0 to 2³² − 1
A word is a w-bit string that may be represented as a sequence of hex digits.

The 32-bit words can be displayed in binary, hex and decimal. To convert a word to hex digits, each 4-bit string is converted to its hex digit equivalent.

In Python, a 32-bit word can be represented using an integer (int). Python supports bitwise operations, such as AND, XOR and bit shifting, which are required by SHA-256. A mask can be used to keep the value within 32 bits.

In [2]:
word = 10
mask = 0xFFFFFFFF

print(bin(word))           # 0b1010
print(hex(word))           # 0xa
print(word)                # 10

word = (word + 5) & mask
print(word)                # 15

0b1010
0xa
10
15


### Sub-section B: Sequences of 32-bit words

Sequences of 32-bit words can be represented using a NumPy array. A 512-bit message block contains 16 words, with each word containing 32 bits (16 × 32 = 512). These words are intermediate data used during SHA-256 processing.

The np.uint32 data type is used to represent unsigned 32-bit integers, ensuring that each word has a fixed size.

In [3]:
words = np.array(
    [0x61626380, 0, 0, 0x00000018],
    dtype=np.uint32
)

print(words)
print(words.dtype)
print(len(words))

[1633837952          0          0         24]
uint32
4


A list is suitable because SHA-256 processes words in a specific order. The dtype property confirms the data type, while len(words) returns the number of words in the array.

### Sub-Section C: Representing input messages

An input message can be represented as a Python string (str) when the input is textual. Before processing, the string must be encoded into a byte sequence because SHA-256 operates on the binary representation of the message rather than directly on Python characters.

Python's encode() method converts text into bytes using a specified character encoding. UTF-8 is a suitable choice for text because it supports a wide range of characters.

In [4]:
message = "abc"
data = message.encode("utf-8")

print(data)       # b'abc'
print(len(data))  # 3

b'abc'
3


The resulting bytes object contains three bytes, corresponding to the ASCII-compatible UTF-8 encodings of a, b and c.

Using bytes is appropriate because it provides an immutable sequence of values in the range 0–255. It also allows the message to be sliced into blocks and processed at the byte level. For non-textual input, the original data can be supplied directly as bytes without first converting it into a string.

The distinction between text and bytes is important because different encodings of the same visible text can produce different byte sequences and therefore different hashes.

### Sub-Section D: Representing 512-bit message blocks

SHA-256 processes the padded message in blocks of 512 bits, equivalent to 64 bytes. The message is padded by appending a single 1 bit, followed by enough 0 bits, and then a 64-bit representation of the original message length. The resulting message length must be a multiple of 512 bits

A Python bytes object is suitable for representing a block because each byte contains eight bits and a complete block can be represented by exactly 64 bytes.

In [5]:
block = bytes(64)

print(len(block))       # 64
print(len(block) * 8)   # 512

64
512


This creates a 64-byte block containing zeros, demonstrating the size of the representation. It is not, by itself, a correctly padded SHA-256 block.

### Sub-Section E: Representing the final 256-bit hash value

SHA-256 produces a fixed-length message digest of 256 bits, regardless of the input message length. This corresponds to 32 bytes, or 64 hexadecimal digits when displayed as text . 

The digest can be represented as a bytes object for binary processing or as a hexadecimal string for presentation.

In [6]:
digest = bytes(32)
hex_digest = digest.hex()

print(len(digest))      # 32
print(len(hex_digest))  # 64

32
64


This example uses zero bytes to demonstrate the output representation; it is not an actual SHA-256 digest.

For verification, Python's hashlib module can produce a reference digest:

In [7]:
reference = hashlib.sha256(b"abc").digest()

print(len(reference))  # 32
print(reference.hex())

32
ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad


In an implementation assignment, hashlib is used only as a reference for testing; the implementation in Problem 6 computes the digest independently. The custom implementation should calculate the digest itself.

Using bytes preserves the exact binary output, while hexadecimal encoding provides a readable representation that can be compared against known test vectors. This supports testing the implementation against published SHA-256 examples.

### Sub-Section F: Big-endian integers

Endianness is the order in which the bytes of a multi-byte integer are stored.
**Big-endian** puts the most significant byte first; **little-endian** puts it last.
FIPS 180-4 (§3.1) specifies big-endian throughout SHA-256: when parsing message bytes
into words, when writing the 64-bit length during padding, and when converting the
final hash words to bytes.

This matters in practice because most modern processors (x86, and ARM in its usual
configuration) are little-endian. Any conversion that relies on the machine's
native order will silently produce wrong words, so every conversion in this
notebook states the byte order explicitly.

In [8]:
print(sys.byteorder)  # 'little' on most machines

n = 0x61626380
print(n.to_bytes(4, "big").hex())     # 61626380: most significant byte first
print(n.to_bytes(4, "little").hex())  # 80636261: reversed

# Reading the same 4 bytes with the wrong order gives a different word
raw = b"abc\x80"
print(hex(int.from_bytes(raw, "big")))     # 0x61626380 (correct for SHA-256)
print(hex(int.from_bytes(raw, "little")))  # 0x80636261 (wrong)

# The same choice in struct and NumPy: '>' means big-endian
print(struct.pack(">I", n).hex())    # 32-bit word
print(struct.pack(">Q", 24).hex())   # 64-bit length field: 0000000000000018

assert np.frombuffer(raw, dtype=">u4")[0] == n
assert np.frombuffer(raw, dtype="<u4")[0] != n

little
61626380
80636261
0x61626380
0x80636261
61626380
0000000000000018


np.frombuffer(raw, dtype=np.uint32) uses native order, so on a little-endian machine it silently gives the wrong word. This is why ">u4" is always specified.

## Problem 2: SHA-256 Bitwise Operations

Two basic operations from FIPS 180-4, Section 3.2, are needed:

- **SHR<sup>n</sup>(x)**, shift right: `x >> n`. Bits pushed off the right end
  are lost, and zeros come in on the left.
- **ROTR<sup>n</sup>(x)**, rotate right: `(x >> n) | (x << (32 - n))`. Bits pushed
  off the right end come back in on the left, so no information is lost.

In [9]:
def rotr(x, n):
    """Rotate a 32-bit unsigned integer right by n bits (ROTR^n in FIPS 180-4).

    Args:
        x: The value to rotate (converted to np.uint32).
        n: Number of bit positions to rotate, 0 < n < 32.

    Returns:
        np.uint32: x rotated right by n bits.
    """
    x = np.uint32(x)
    n = np.uint32(n)
    # Right part: bits that move right. Left part: bits that wrapped around.
    return np.uint32((x >> n) | (x << (np.uint32(32) - n)))


# Tests
assert rotr(0x00000001, 1) == 0x80000000  # lowest bit wraps to the top
assert rotr(0x12345678, 4) == 0x81234567  # one hex digit = 4 bits
assert isinstance(rotr(1, 1), np.uint32)
print("rotr: all tests passed")

rotr: all tests passed


What’s going on: Take 0x12345678 and rotate it by 4 bits. One hex digit is 4 bits, so the last digit 8 moves to the front: 0x81234567. x << (32-n) would go past 32 bits in normal Python, but uint32 throws away anything above bit 31, so only the wrapped bits are left. The | joins the two halves.

The shift amount is also wrapped in np.uint32 so the result stays uint32 on every NumPy version. Mixing it with a plain Python int can turn the result into int64 on older versions.

### 1. Ch(x, y, z): Choose

$$Ch(x, y, z) = (x \land y) \oplus (\lnot x \land z)$$

For each bit position, `x` acts as a selector:
- if the bit of `x` is **1**, the output takes the bit of **y**
- if the bit of `x` is **0**, the output takes the bit of **z**

It works like an `if x then y else z` applied separately to each of the 32 bits.
`x & y` keeps y's bits where x is 1, and `~x & z` keeps z's bits where x is 0.
The two never overlap, so XOR combines them.

In SHA-256, Ch is applied to the working variables e, f and g in every round.

In [10]:
def Ch(x, y, z):
    """Choose function from FIPS 180-4 (Equation 4.2).

    For each bit, select the bit of y where x is 1, and the bit of z where x is 0.

    Args:
        x, y, z: 32-bit words (converted to np.uint32).

    Returns:
        np.uint32: (x AND y) XOR (NOT x AND z).
    """
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    return np.uint32((x & y) ^ (~x & z))


# Tests
# Single-bit truth table: the result should be y when x=1 and z when x=0.
for a in (0, 1):
    for b in (0, 1):
        for c in (0, 1):
            assert Ch(a, b, c) == (b if a else c)

# x all 1s means take all of y. x all 0s means take all of z.
assert Ch(0xFFFFFFFF, 0xAAAAAAAA, 0x55555555) == 0xAAAAAAAA
assert Ch(0x00000000, 0xAAAAAAAA, 0x55555555) == 0x55555555

# Mixed selector: top half from y, bottom half from z.
assert Ch(0xFFFF0000, 0x12345678, 0x9ABCDEF0) == 0x1234DEF0
print("Ch: all tests passed")

Ch: all tests passed


What’s going on: The last test shows the idea best. x = 0xFFFF0000 is 1s in the top half and 0s in the bottom half. So the top half comes from y (1234) and the bottom half comes from z (DEF0). ~x flips every bit, and on a uint32 that gives exactly 32 flipped bits. A plain Python int would give a negative number instead, which is a big reason to use NumPy here.

### 2. Maj(x, y, z): Majority

$$Maj(x, y, z) = (x \land y) \oplus (x \land z) \oplus (y \land z)$$

For each bit position, the output is whichever value (0 or 1) appears in
**at least two** of x, y and z, like a 2-out-of-3 vote.

Why the formula works: if at least two bits are 1, either exactly one of the
three AND pairs is 1, or all three are (1 ⊕ 1 ⊕ 1 = 1). Either way the XOR
gives 1. If fewer than two bits are 1, every pair is 0.

In SHA-256, Maj is applied to the working variables a, b and c in every round.

In [11]:
def Maj(x, y, z):
    """Majority function from FIPS 180-4 (Equation 4.3).

    For each bit, output the value held by at least two of x, y and z.

    Args:
        x, y, z: 32-bit words (converted to np.uint32).

    Returns:
        np.uint32: (x AND y) XOR (x AND z) XOR (y AND z).
    """
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    return np.uint32((x & y) ^ (x & z) ^ (y & z))


# Tests
# Single-bit truth table: the result is 1 when two or more inputs are 1.
for a in (0, 1):
    for b in (0, 1):
        for c in (0, 1):
            assert Maj(a, b, c) == (1 if a + b + c >= 2 else 0)

assert Maj(0xFFFFFFFF, 0xFFFFFFFF, 0x00000000) == 0xFFFFFFFF  # two of three are 1
assert Maj(0xFFFFFFFF, 0x00000000, 0x00000000) == 0x00000000  # only one is 1

# Mixed: in each hex digit, 2 of 3 inputs agree with y.
assert Maj(0xF0F0F0F0, 0xFF00FF00, 0x0F0F0F0F) == 0xFF00FF00
print("Maj: all tests passed")

Maj: all tests passed


What’s going on: Look at the first hex digit of the last test: x=F, y=F, z=0. Two 1s win, so the output is F. The second digit: x=0, y=F, z=F, so again F. The third: x=F, y=0, z=0, so 0. That pattern gives FF00FF00.

### 3. Sigma0(x)

$$\Sigma_0^{256}(x) = ROTR^{2}(x) \oplus ROTR^{13}(x) \oplus ROTR^{22}(x)$$

**What it does:** x is rotated by three different amounts (2, 13 and 22 bits)
and the three results are XORed together.

**Why:** every output bit now depends on three input bits from different places
in the word, so changing one input bit changes three output bits. This spreading
effect is called **diffusion**, and it is what makes a small change in the
message produce a completely different hash. Only rotations are used, so no
information is lost.

**Where it is used:** on working variable a in every round.

**Tests:** using x = 1 (a single 1 bit at position 0) makes the expected result
easy to work out by hand. Rotating right by n moves that bit to position 32 − n,
so it should land at bits 30, 19 and 10, which gives `0x40080400`. Zero should
give zero.

In [12]:
def Sigma0(x):
    """Big sigma 0 (FIPS 180-4, Eq. 4.4): ROTR^2(x) XOR ROTR^13(x) XOR ROTR^22(x)."""
    x = np.uint32(x)
    return np.uint32(rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22))


assert Sigma0(0) == 0
# Bit 0 lands at bits 30, 19 and 10.
assert Sigma0(1) == 0x40080400
print("Sigma0: tests passed")

Sigma0: tests passed


### 4. Sigma1(x)

$$\Sigma_1^{256}(x) = ROTR^{6}(x) \oplus ROTR^{11}(x) \oplus ROTR^{25}(x)$$

**What it does:** the same structure as Σ₀, three rotations XORed together, but
with rotation amounts of 6, 11 and 25.

**Why different amounts:** Σ₁ is applied to a different part of the state
(variable e instead of a). Using different amounts means the two halves of the
state are mixed in different ways, which gives better overall diffusion.

**Where it is used:** on working variable e in every round.

**Tests:** with x = 1, the bit should land at positions 32 − 6 = 26,
32 − 11 = 21 and 32 − 25 = 7, which gives `0x04200080`.

In [13]:
def Sigma1(x):
    """Big sigma 1 (FIPS 180-4, Eq. 4.5): ROTR^6(x) XOR ROTR^11(x) XOR ROTR^25(x)."""
    x = np.uint32(x)
    return np.uint32(rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25))


assert Sigma1(0) == 0
# Bit 0 lands at bits 26, 21 and 7.
assert Sigma1(1) == 0x04200080
print("Sigma1: tests passed")

Sigma1: tests passed


### 5. sigma0(x)

$$\sigma_0^{256}(x) = ROTR^{7}(x) \oplus ROTR^{18}(x) \oplus SHR^{3}(x)$$

**What it does:** nearly the same as the big sigmas, with one key difference:
the third term is a **shift**, not a rotation. The 3 bits that fall off the end
are lost and replaced with zeros.

**Why it matters:** because information is thrown away, the function cannot be
reversed. This gives the message schedule a different kind of mixing from the
compression rounds.

**Where it is used:** in the **message schedule**, which expands each 16-word
block of the message to 64 words:
$W_t = \sigma_1(W_{t-2}) + W_{t-7} + \sigma_0(W_{t-15}) + W_{t-16}$.

**Tests:** the most useful input is all 1s (`0xFFFFFFFF`). The two rotations
both give all 1s, which cancel out (1 ⊕ 1 = 0), so only the shift is left:
`0xFFFFFFFF >> 3 = 0x1FFFFFFF`. The big sigmas returned all 1s for the same
input, so this test confirms the third term really is a shift.

In [14]:
def sigma0(x):
    """Small sigma 0 (FIPS 180-4, Eq. 4.6): ROTR^7(x) XOR ROTR^18(x) XOR SHR^3(x)."""
    x = np.uint32(x)
    return np.uint32(rotr(x, 7) ^ rotr(x, 18) ^ (x >> np.uint32(3)))


assert sigma0(0) == 0
# The rotations cancel, leaving only SHR^3 (top 3 bits are zero).
assert sigma0(0xFFFFFFFF) == 0x1FFFFFFF
print("sigma0: tests passed")

sigma0: tests passed


### 6. sigma1(x)

$$\sigma_1^{256}(x) = ROTR^{17}(x) \oplus ROTR^{19}(x) \oplus SHR^{10}(x)$$

**What it does:** the same structure as σ₀, two rotations and one shift, with
amounts 17, 19 and 10.

**Where it is used:** also in the message schedule, applied to $W_{t-2}$. Together,
σ₀ and σ₁ make sure every new word depends on several earlier words in a
non-linear way.

**Tests:** with all 1s, the rotations cancel again, leaving only
`0xFFFFFFFF >> 10 = 0x003FFFFF` (the top 10 bits are zero).

### Summary
| Function | Built from | Used in |
|---|---|---|
| Ch, Maj | per-bit logic | compression rounds |
| Σ₀, Σ₁ | three rotations | compression rounds (a and e) |
| σ₀, σ₁ | two rotations and a shift | message schedule |

**Reference:** NIST, *FIPS 180-4: Secure Hash Standard*, Section 4.1.2.

In [15]:
def sigma1(x):
    """Small sigma 1 (FIPS 180-4, Eq. 4.7): ROTR^17(x) XOR ROTR^19(x) XOR SHR^10(x)."""
    x = np.uint32(x)
    return np.uint32(rotr(x, 17) ^ rotr(x, 19) ^ (x >> np.uint32(10)))


assert sigma1(0) == 0
# The rotations cancel, leaving only SHR^10 (top 10 bits are zero).
assert sigma1(0xFFFFFFFF) == 0x003FFFFF
print("sigma1: tests passed")

sigma1: tests passed


## Problem 4: Padding and Parsing Messages

SHA-256 reads its input in 512-bit blocks, so a message is padded and then split up
before any hashing happens [1]. This problem is built one requirement at a time:

1. append the `1` bit,
2. append the zero bits,
3. append the original length as a 64-bit integer,
4. handle messages that need one or two final blocks,
5. read the message in the required byte order,
6. put it together as the generator `block_parse`,
7. test it, including the boundary cases.

Steps 1 to 4 are [section 5.1.1 of FIPS 180-4][1] (padding) and step 5 is
[section 5.2.1][1] (parsing). Each step is shown on the standard's own example, the
8-bit ASCII message `"abc"`.

[1]: https://doi.org/10.6028/NIST.FIPS.180-4

### Step 1: Append the `1` bit

[Section 5.1.1][1] says to append a single `1` bit to the message. The input is a
`bytes` object, so the message is always a whole number of bytes. A `1` bit followed by
seven `0` bits is the single byte `0b10000000 = 0x80`, so appending the bit means
appending that byte. The original length `l` (in bits) is saved first, because the
padding changes the length and step 3 needs the original.

[1]: https://doi.org/10.6028/NIST.FIPS.180-4

In [16]:
msg = b"abc"                # worked example from section 5.1.1
bit_len = len(msg) * 8      # l, the original length in bits (24 here)

padded = msg + b"\x80"      # the 1 bit, followed by seven 0 bits to fill the byte
print(padded)

b'abc\x80'


### Step 2: Append the zero bits

[Section 5.1.1][1] says to append `k` zero bits, where `k` is the smallest non-negative
solution of `l + 1 + k ≡ 448 (mod 512)`. The 448 leaves exactly 64 bits at the end of
the block for the length. In bytes, 448 bits is 56 bytes and 512 bits is 64, so the rule
is: add zero bytes until the length is 56 (mod 64). The `1` bit was already added in
step 1, so the `+ 1` in the equation is already included in `len(padded)`.

The `% 64` is what makes this correct whichever side of 56 the length is on. If
`len(padded)` is below 56 (mod 64), `56 - len(padded)` is the number of zeros needed. If
it is already past 56, the subtraction goes negative and `% 64` wraps it round to the
number of zeros needed to reach 56 in the next block.

[1]: https://doi.org/10.6028/NIST.FIPS.180-4

In [17]:
# Pad with zero bytes until the length is 56 (mod 64).
padded += b"\x00" * ((56 - len(padded)) % 64)

print(len(padded), len(padded) % 64)   # 56 56

56 56


### Step 3: Append the original length

The last step of [section 5.1.1][1] appends `l`, the length of the *original* message in
bits, as a 64-bit integer. That is 8 bytes, which is why step 2 stopped at 56 bytes
(56 + 8 = 64). The standard uses a big-endian convention ([section 3.1][1]), so the
most significant byte comes first. Python's [`int.to_bytes`][2] does the conversion.

[1]: https://doi.org/10.6028/NIST.FIPS.180-4
[2]: https://docs.python.org/3/library/stdtypes.html#int.to_bytes

In [18]:
# 64-bit big-endian length of the original message, in bits.
padded += bit_len.to_bytes(8, byteorder="big")

print(len(padded))   # 64, so "abc" pads to exactly one 512-bit block

64


### Step 4: One or two final blocks

Steps 1 to 3 together add at least 9 bytes: the `0x80` byte and the 8-byte length. So
the number of blocks depends on how much room the last block has:

| message length | blocks | what happens in the last block |
|---|---|---|
| 0 to 55 bytes | 1 | `0x80`, zeros, length all fit |
| 56 to 63 bytes | 2 | `0x80` and zeros fill block 1, block 2 is zeros plus length |
| 64 bytes | 2 | block 1 is the message, block 2 is `0x80`, zeros, length |
| 65 to 119 bytes | 2 | same as the first row, one block later |
| 120 bytes | 3 | same as the second row, one block later |

No special case is needed in the code. The `% 64` from step 2 already produces the extra
block when the length does not fit. Here the three steps are combined into one function,
`pad`, and checked at the boundaries.

In [19]:
def pad(msg):
    """Return msg padded to a multiple of 64 bytes (512 bits), as in section 5.1.1."""
    bit_len = len(msg) * 8                                  # l, in bits
    padded = bytes(msg) + b"\x80"                           # step 1: the 1 bit
    padded += b"\x00" * ((56 - len(padded)) % 64)           # step 2: k zero bits
    padded += bit_len.to_bytes(8, byteorder="big")          # step 3: 64-bit length
    return padded


lengths = (0, 55, 56, 63, 64, 119, 120)
for n in lengths:
    print(n, "bytes ->", len(pad(bytes(n))) // 64, "block(s)")

assert [len(pad(bytes(n))) // 64 for n in lengths] == [1, 1, 2, 2, 2, 2, 3]

0 bytes -> 1 block(s)
55 bytes -> 1 block(s)
56 bytes -> 2 block(s)
63 bytes -> 2 block(s)
64 bytes -> 2 block(s)
119 bytes -> 2 block(s)
120 bytes -> 3 block(s)


### Step 5: Byte order

[Section 5.2.1][1] splits the padded message into 512-bit blocks, each read as sixteen
32-bit words, word 0 first. The standard is big-endian ([section 3.1][1]), so the first
byte of each group of four is the most significant byte of the word. NumPy's
[`frombuffer`][4] can read the bytes straight into integers, and the dtype `">u4"` means
"big-endian unsigned 32-bit" ([byte-swapping][5]). Reading with the wrong order (`"<u4"`,
little-endian, which is what most CPUs use natively) gives different numbers, so the
check below compares them.

[1]: https://doi.org/10.6028/NIST.FIPS.180-4
[4]: https://numpy.org/doc/stable/reference/generated/numpy.frombuffer.html
[5]: https://numpy.org/doc/stable/user/basics.byteswapping.html

In [20]:
block = pad(b"abc")[:64]

words_be = np.frombuffer(block, dtype=">u4")   # big-endian, as the standard requires
words_le = np.frombuffer(block, dtype="<u4")   # little-endian, the wrong choice here

print(f"{words_be[0]:08x} vs {words_le[0]:08x}")   # 61626380 vs 80636261
print(words_be[15], "vs", words_le[15])            # 24 vs 402653184

assert words_be[0] == 0x61626380 and words_be[15] == 24

61626380 vs 80636261
24 vs 402653184


### Step 6: The generator

`block_parse` pads the message with `pad`, then yields it one block at a time as sixteen
`uint32` values. Writing it as a [generator][3] means the hash computation in the next
problem can loop with `for block in block_parse(msg)` and only ever hold one block.
`.astype(np.uint32)` converts from the big-endian dtype to native `uint32`, so later
arithmetic and bit operations behave normally.

[3]: https://docs.python.org/3/reference/expressions.html#yield-expressions

In [21]:
def block_parse(msg):
    """Yield the padded 512-bit blocks of msg as arrays of sixteen uint32 words.

    Padding follows section 5.1.1 of FIPS 180-4 and parsing follows section 5.2.1:
    https://doi.org/10.6028/NIST.FIPS.180-4
    """
    if not isinstance(msg, (bytes, bytearray)):
        raise TypeError("msg must be a bytes object")

    padded = pad(msg)    # section 5.1.1: the 1 bit, the zero bits and the length

    # Section 5.2.1: split into 512-bit (64-byte) blocks of sixteen 32-bit words.
    for i in range(0, len(padded), 64):
        # ">u4" reads each group of 4 bytes as a big-endian unsigned 32-bit integer.
        yield np.frombuffer(padded[i:i + 64], dtype=">u4").astype(np.uint32)


for block in block_parse(b"abc"):
    print([f"{w:08x}" for w in block])

['61626380', '00000000', '00000000', '00000000', '00000000', '00000000', '00000000', '00000000', '00000000', '00000000', '00000000', '00000000', '00000000', '00000000', '00000000', '00000018']


### Test 1: The standard's example

The worked example in [section 5.1.1][1] pads `"abc"` to one block that starts with
`0x61626380` and ends with the length, 24.

[1]: https://doi.org/10.6028/NIST.FIPS.180-4

In [22]:
b = list(block_parse(b"abc"))
assert len(b) == 1 and b[0].shape == (16,) and b[0].dtype == np.uint32
assert b[0][0] == 0x61626380   # 'a' 'b' 'c' then the 0x80 byte, big-endian
assert not b[0][1:15].any()    # zero padding
assert b[0][15] == 24          # l = 24 bits

### Test 2: The empty message

With no message, the only content is the `1` bit and a length of zero, so the block
starts with `0x80000000` and is zero everywhere else.

In [23]:
b = list(block_parse(b""))
assert len(b) == 1 and b[0][0] == 0x80000000 and not b[0][1:].any()

### Test 3: Boundary lengths

55 and 56 bytes are where the length stops fitting in the final block. 64 is an exact
block, so the padding gets a block to itself. 119 and 120 are the same boundary one
block later. For each length the padded bytes are rebuilt from the words and every part
of section 5.1.1 is checked separately: the message, the `1` bit, the zero bits, and the
64-bit length.

for n, expected in [(0, 1), (1, 1), (55, 1), (56, 2), (57, 2), (63, 2), (64, 2),
                    (119, 2), (120, 3), (128, 3), (1000, 16)]:
    msg = (bytes(range(256)) * 4)[:n]
    blocks = list(block_parse(msg))
    assert len(blocks) == expected, (n, len(blocks))
    assert all(blk.shape == (16,) and blk.dtype == np.uint32 for blk in blocks)

    # Rebuild the padded bytes from the words and check each part of section 5.1.1.
    raw = b"".join(int(w).to_bytes(4, "big") for blk in blocks for w in blk)
    assert len(raw) % 64 == 0                          # whole number of 512-bit blocks
    assert raw[:n] == msg                              # message, byte order intact
    assert raw[n] == 0x80                              # the 1 bit
    assert not any(raw[n + 1:-8])                      # the zero bits
    assert int.from_bytes(raw[-8:], "big") == n * 8    # 64-bit length

### Test 4: Exact positions at 55, 56 and 64 bytes

The most error-prone lengths, checked word by word: at 55 bytes the `0x80` byte is the
last thing before the length, at 56 bytes the length spills into a second block, and at
64 bytes the second block holds only the padding.

In [24]:
# 55 bytes: the 0x80 byte lands in the last byte before the length field.
b = list(block_parse(b"a" * 55))
assert b[0][13] == 0x61616180 and b[0][14] == 0 and b[0][15] == 440

# 56 bytes: the length no longer fits, so block 2 is zeros followed by the length.
b = list(block_parse(b"a" * 56))
assert b[0][14] == 0x80000000 and b[0][15] == 0
assert not b[1][:15].any() and b[1][15] == 448

# 64 bytes: the padding sits entirely in the second block.
b = list(block_parse(b"a" * 64))
assert b[1][0] == 0x80000000 and b[1][15] == 512

### Test 5: Input handling

`block_parse` is a generator, `bytearray` is accepted as well as `bytes`, and anything
that is not bytes-like raises `TypeError`.

In [25]:
assert isinstance(block_parse(b""), types.GeneratorType)
assert [list(x) for x in block_parse(bytearray(b"abc"))] == [list(x) for x in block_parse(b"abc")]

for bad in ["abc", 123, None, [1, 2, 3]]:
    try:
        list(block_parse(bad))
    except TypeError:
        pass
    else:
        raise AssertionError(f"{bad!r} should have raised TypeError")

print("All tests passed.")

All tests passed.


### Comparison with other implementations

This version builds the whole padded message in memory before yielding anything. That
is simple and matches the wording of the standard closely, but it uses memory
proportional to the message size. Real libraries avoid that. Python's
[`hashlib`][6] accepts data in chunks through `update()` and only applies the padding
when the digest is requested, so it can hash data that does not fit in memory. The C
reference code in [RFC 6234][7] works the same way, buffering one 64-byte block and
padding it at the end. A streaming version of `block_parse` could do the same by
yielding full blocks as they arrive and only padding the final partial one.

`hashlib` is also the check for the next problems: once the compression function is
written, the digest of the same message can be compared against `hashlib.sha256`.

[6]: https://docs.python.org/3/library/hashlib.html
[7]: https://www.rfc-editor.org/rfc/rfc6234